In [5]:
import numpy as np
import seaborn as sns
import pandas as pd
import folium
import openrouteservice as ors


In [6]:
ORSkey = 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjZjYWQ2ZmE1MTczMzRjZjA5YjhlODE4Mzg2MTZmZmUzIiwiaCI6Im11cm11cjY0In0='

In [7]:
client = ors.Client(key=ORSkey)

In [8]:
locations = pd.read_csv("project_data/FoodstuffsLocations.csv")

routes = pd.read_csv(
    "project_data/Scheduled_Optimal_Routes.csv"
)

routes.head()

,Day,Truck,Scheduled_Shift,Start_Time,End_Time,Overtime_Minutes,Route,Demand,Duration_hours,Vehicle_Type,Trip_Cost_NZD
0,Saturday,Truck 2,2pm,14:00,17:35,0.0,Warehouse -> New World Mt Albert -> New World ...,10.0,3.585756,Owned,788.87
1,Saturday,Truck 3,2pm,14:00,17:30,0.0,Warehouse -> Pak 'n Save Royal Oak -> Four Squ...,10.0,3.493192,Owned,768.50
2,Saturday,Truck 4,2pm,14:00,17:29,0.0,Warehouse -> Pak 'n Save Sylvia Park -> Warehouse,8.0,3.491542,Owned,768.14
3,Saturday,Truck 5,2pm,14:00,17:09,0.0,Warehouse -> Pak 'n Save Clendon -> Warehouse,7.0,3.157594,Owned,694.67
4,Saturday,Truck 6,2pm,14:00,17:03,0.0,Warehouse -> Pak 'n Save Mangere -> Warehouse,8.0,3.044325,Owned,669.75


In [9]:
routes["Demand"] = pd.to_numeric(
    routes["Demand"],
    errors="coerce"
)

routes["Duration_hours"] = pd.to_numeric(
    routes["Duration_hours"],
    errors="coerce"
)

In [10]:
routes["Demand"] = pd.to_numeric(
    routes["Demand"],
    errors="coerce"
)

routes["Duration_hours"] = pd.to_numeric(
    routes["Duration_hours"],
    errors="coerce"
)


In [11]:
location_coords = dict(
    zip(
        locations["Supermarket"].str.strip(),
        zip(
            locations["Long"],
            locations["Lat"]
        )
    )
)

In [12]:
missing_locations = set()

for route_string in routes["Route"].dropna():

    stops = [
        stop.strip()
        for stop in route_string.split("->")
    ]

    for stop in stops:

        if stop not in location_coords:
            missing_locations.add(stop)

if missing_locations:

    print("Missing locations:")

    for location in sorted(missing_locations):
        print("-", location)

else:

    print("All route locations were found.")

All route locations were found.


In [13]:
def create_base_map():
    warehouse = locations[
        locations["Type"] == "Warehouse"
        ]

    if warehouse.empty:

        map_location = [
            locations.iloc[0]["Lat"],
            locations.iloc[0]["Long"]
        ]

    else:

        map_location = [
            warehouse.iloc[0]["Lat"],
            warehouse.iloc[0]["Long"]
        ]

    map_obj = folium.Map(
        location=map_location,
        zoom_start=10.25
    )

    # --------------------------------------------------------
    # Add store markers
    # --------------------------------------------------------

    for _, location in locations.iterrows():

        if location["Type"] == "Four Square":

            icon_colour = "green"

        elif location["Type"] == "New World":

            icon_colour = "red"

        elif location["Type"] == "Pak 'n Save":

            icon_colour = "orange"

        elif location["Type"] == "Warehouse":

            icon_colour = "black"

        folium.Marker(
            [
                location["Lat"],
                location["Long"]
            ],
            popup=location["Supermarket"],
            icon=folium.Icon(
                color=icon_colour
            )
        ).add_to(map_obj)

    return map_obj

In [14]:
route_colours = [
    "blue",
    "purple",
    "darkred",
    "darkgreen",
    "cadetblue",
    "pink",
    "black",
    "gray",
    "orange",
    "lightblue",
    "red",
    "green",
    "beige",
    "darkblue",
    "darkpurple",
    "lightgreen",
    "lightred"
]

In [15]:
import time
import openrouteservice.exceptions

# Store ORS results so the same route is never requested twice
route_cache = {}


def add_routes_to_map(
        map_obj,
        routes_to_plot
):
    for route_num, (_, row) in enumerate(
            routes_to_plot.iterrows()
    ):

        # ----------------------------------------------------
        # Get stops
        # ----------------------------------------------------

        stops = [
            stop.strip()
            for stop in row["Route"].split("->")
        ]

        # ----------------------------------------------------
        # Check locations
        # ----------------------------------------------------

        if any(
                stop not in location_coords
                for stop in stops
        ):
            print(
                f"Skipping route {route_num + 1}: "
                f"location not found"
            )

            continue

        # ----------------------------------------------------
        # Create a unique key for this route
        # ----------------------------------------------------

        route_key = tuple(stops)

        # ----------------------------------------------------
        # Use cached ORS result if available
        # ----------------------------------------------------

        if route_key in route_cache:

            route_data = route_cache[route_key]

        else:

            print(
                f"Requesting ORS route "
                f"{route_num + 1}..."
            )

            route_coords = [
                list(location_coords[stop])
                for stop in stops
            ]

            # ------------------------------------------------
            # Request route from ORS
            # ------------------------------------------------

            try:

                route_data = client.directions(

                    coordinates=route_coords,

                    profile="driving-hgv",

                    format="geojson",

                    validate=False

                )

            except openrouteservice.exceptions._OverQueryLimit:

                print(
                    "ORS rate limit reached. "
                    "Stopping route requests."
                )

                break

            # ------------------------------------------------
            # Save result in cache
            # ------------------------------------------------

            route_cache[route_key] = route_data

            # Small pause between API requests
            time.sleep(1)

        # ----------------------------------------------------
        # Extract ORS information
        # ----------------------------------------------------

        feature = route_data["features"][0]

        distance = (
            feature["properties"]
            ["summary"]
            ["distance"]
        )

        duration = (
            feature["properties"]
            ["summary"]
            ["duration"]
        )

        duration_hours = duration / 3600

        geometry = (
            feature["geometry"]
            ["coordinates"]
        )

        # ----------------------------------------------------
        # Colour
        # ----------------------------------------------------

        colour = route_colours[
            route_num % len(route_colours)
            ]

        # ----------------------------------------------------
        # Add route to map
        # ----------------------------------------------------

        folium.PolyLine(

            locations=[
                list(reversed(coord))
                for coord in geometry
            ],

            color=colour,

            weight=5,

            opacity=0.8,

            tooltip=(

                f"Truck: {row['Truck']} | "

                f"Shift: "
                f"{row['Scheduled_Shift']} | "

                f"Day: {row['Day']} | "

                f"Demand: "
                f"{row['Demand']:.0f} pallets | "

                f"Distance: "
                f"{distance / 1000:.2f} km | "

                f"ORS Duration: "
                f"{duration_hours:.2f} hrs"

            )

        ).add_to(map_obj)

In [16]:
routes_8am = routes[
    routes["Scheduled_Shift"]
    .astype(str)
    .str.strip()
    .str.lower()
    == "8am"
    ].copy()

print(
    f"Number of 8am routes: {len(routes_8am)}"
)

routes_2pm = routes[
    routes["Scheduled_Shift"]
    .astype(str)
    .str.strip()
    .str.lower()
    == "2pm"
    ].copy()

print(
    f"Number of 2pm routes: {len(routes_2pm)}"
)

Number of 8am routes: 40
Number of 2pm routes: 20


In [17]:
# Clean the Day and Shift values

day = (
    routes["Day"]
    .astype(str)
    .str.strip()
    .str.lower()
)

shift = (
    routes["Scheduled_Shift"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# ============================================================
# WEEKDAYS
# ============================================================

weekdays_8am = routes[
    (day == "weekdays")
    &
    (shift == "8am")
    ].copy()

weekdays_2pm = routes[
    (day == "weekdays")
    &
    (shift == "2pm")
    ].copy()

# ============================================================
# SATURDAYS
# ============================================================

saturdays_8am = routes[
    day.isin(["saturday", "saturdays"])
    &
    (shift == "8am")
    ].copy()

saturdays_2pm = routes[
    day.isin(["saturday", "saturdays"])
    &
    (shift == "2pm")
    ].copy()

# ============================================================
# CHECK
# ============================================================

print("Weekdays 8am:", len(weekdays_8am))
print("Weekdays 2pm:", len(weekdays_2pm))
print("Saturdays 8am:", len(saturdays_8am))
print("Saturdays 2pm:", len(saturdays_2pm))

Weekdays 8am: 20
Weekdays 2pm: 15
Saturdays 8am: 20
Saturdays 2pm: 5


In [18]:
map_weekdays_8am = create_base_map()

add_routes_to_map(
    map_weekdays_8am,
    weekdays_8am
)

map_weekdays_8am

Requesting ORS route 1...
Requesting ORS route 2...
Requesting ORS route 3...
Requesting ORS route 4...
Requesting ORS route 5...
Requesting ORS route 6...
Requesting ORS route 7...
Requesting ORS route 8...
Requesting ORS route 9...
Requesting ORS route 10...
Requesting ORS route 11...
Requesting ORS route 12...
Requesting ORS route 13...
Requesting ORS route 14...
Requesting ORS route 15...
Requesting ORS route 16...
Requesting ORS route 17...
Requesting ORS route 18...
Requesting ORS route 19...
Requesting ORS route 20...


In [19]:
map_weekdays_2pm = create_base_map()

add_routes_to_map(
    map_weekdays_2pm,
    weekdays_2pm
)

map_weekdays_2pm

Requesting ORS route 1...
Requesting ORS route 2...
Requesting ORS route 3...
Requesting ORS route 4...
Requesting ORS route 5...
Requesting ORS route 6...
Requesting ORS route 7...
Requesting ORS route 8...
Requesting ORS route 9...
Requesting ORS route 10...
Requesting ORS route 11...
Requesting ORS route 12...
Requesting ORS route 13...
Requesting ORS route 14...
Requesting ORS route 15...


In [20]:
map_saturdays_8am = create_base_map()

add_routes_to_map(
    map_saturdays_8am,
    saturdays_8am
)

map_saturdays_8am

Requesting ORS route 1...
Requesting ORS route 2...
Requesting ORS route 4...
Requesting ORS route 5...
Requesting ORS route 6...
Requesting ORS route 7...
Requesting ORS route 8...
Requesting ORS route 9...
Requesting ORS route 10...
Requesting ORS route 11...
Requesting ORS route 12...
Requesting ORS route 14...
Requesting ORS route 15...
Requesting ORS route 16...
Requesting ORS route 17...
Requesting ORS route 18...
Requesting ORS route 19...
Requesting ORS route 20...


In [21]:
map_saturdays_2pm = create_base_map()

add_routes_to_map(
    map_saturdays_2pm,
    saturdays_2pm
)

map_saturdays_2pm

Requesting ORS route 1...
Requesting ORS route 2...


In [22]:
map_weekdays_8am.save(
    "project_data/maps/Optimal_Routes_Weekdays_8AM.html"
)

map_weekdays_2pm.save(
    "project_data/maps/Optimal_Routes_Weekdays_2PM.html"
)

map_saturdays_8am.save(
    "project_data/maps/Optimal_Routes_Saturdays_8AM.html"
)

map_saturdays_2pm.save(
    "project_data/maps/Optimal_Routes_Saturdays_2PM.html"
)